# Lamoda · покупательские группы и проверка прогноза на истории клиента

Запустите все ячейки из корня репозитория. Ноутбук создаёт `submission_quarters.csv` для Kaggle и контрольный `submission_context.csv`. Положите ZIP соревнования рядом с ноутбуком или задайте путь через `LAMODA_DATA`. Половину веса обеих моделей получает прежняя посылка с public AP@20% 0,77290.

In [1]:
%run ./lamoda_motifs.ipynb

               size     mean
scoring_date                
2022-01-01    25000  0.34764
2022-02-01    25000  0.33368
2022-03-01    25000  0.33924


Матрицы признаков: (75000, 29) (25000, 29)


699 признаков


Переходы и все признаки: (75000, 733) (75000, 861)


date                  2022-02-01  2022-03-01     mean
model                                                
combined_regional        0.76002     0.77011  0.76506
full_regional            0.75932     0.76985  0.76458
transitions_regional     0.75930     0.76812  0.76371
transitions              0.75465     0.76282  0.75874
regional                 0.74834     0.76359  0.75597
Выбрано по средней AP@20%: combined_regional
Проверены 25,000 строк и порядок клиентов по образцу Kaggle.


Новых признаков 459


      date    model     ap20  buyers_in_top_5000
2022-02-01 previous 0.760016                4114
2022-02-01   motifs 0.774831                4159
2022-03-01 previous 0.770113                4142
2022-03-01   motifs 0.782566                4169


25000 клиентов в submission_motifs.csv


### Два новых прогноза

Первая модель обучает семь бустингов. Вес каждого клиента в обучении зависит от его вероятности принадлежать к группе с похожей частотой покупок, давностью и историей категорий. Вторая модель для каждого порога поддержки (1, 2, 3 прошлых совпадения) выбирает самую длинную из повторявшихся последовательностей. Оба метода используют одинаковые 110 исходных признаков, отобранных на обучающем срезе.

In [2]:
from research.behavior_experts import gate_features, learned_experts
from research.context_backoff import backoff_features
from research.rerank import compare
from zipfile import ZipFile, ZipInfo, ZIP_DEFLATED

core = pd.concat([core_train, core_test])
history = extra_motifs.filter(regex='^motif_mode_').reindex(core.index)
groups = gate_features(core, history)
backoff = backoff_features(core, history)
assert core.index.is_unique and history.shape[1] == 144

def context_forecast(fit_ids, query_ids, prior, personal=None):
    target = labels.loc[fit_ids]
    chosen = list(select_by_gain(core.loc[fit_ids], target, 110))
    features = core[chosen].join(history)
    digest = hashlib.sha256(b'context-ensemble-v1')
    for filename in ['research/behavior_experts.py', 'research/context_backoff.py', 'research/window_models.py']:
        digest.update(Path(filename).read_bytes())
    for package in ['numpy','pandas','scikit-learn']:
        digest.update(version(package).encode())
    for obj in [features, groups, backoff, target, pd.Series(query_ids).reset_index(drop=True)]:
        digest.update(repr(getattr(obj,'columns',None)).encode())
        digest.update(pd.util.hash_pandas_object(obj, index=True).to_numpy().tobytes())
    key = 'context_' + digest.hexdigest()[:24]
    by_group = cached_forecast(key+'_mixture', query_ids, lambda: learned_experts(
        features.loc[fit_ids], target, features.loc[query_ids], groups.loc[fit_ids], groups.loc[query_ids]))
    augmented = features.join(backoff)
    if personal is not None:
        augmented = augmented.join(personal, validate='one_to_one')
        digest.update(Path('research/personal_backtest.py').read_bytes())
        digest.update(version('scipy').encode())
        digest.update(repr(list(personal.columns)).encode())
        digest.update(pd.util.hash_pandas_object(personal, index=True).to_numpy().tobytes())
        key = 'quarter_' + digest.hexdigest()[:24]
    by_suffix = cached_forecast(key+'_backoff', query_ids, lambda: forecast(
        augmented.loc[fit_ids], target, augmented.loc[query_ids]))
    return pd.Series(.5*np.asarray(prior) + .25*(by_group+by_suffix), index=query_ids, name='score')

### Как выбрать формулу по предыдущим кварталам клиента

Для каждого клиента сравниваем шесть вариантов прогноза следующей даты, включая медианный интервал всех заказов, медиану последних трёх интервалов и цикл ведущей категории. Каждые 30 дней в прошлом строим прогноз с обычной слепой зоной. Для проверки берём полностью наблюдаемый период из 90 дней. Наибольший вес получают примеры с давностью последнего заказа, близкой к текущей. Средняя квадратичная ошибка каждой формулы и её систематическое отклонение поступают в бустинг. Границы и длина окна одинаковы для всех четырёх когорт.

In [3]:
from research.personal_backtest import personal_backtest
from research.sequence_rules import baskets

customers = pd.concat([train[['customer_id', 'scoring_date']], test[['customer_id', 'scoring_date']]])
snapshots = []
for _, cohort in customers.groupby('scoring_date'):
    receipts = baskets(orders, cohort)
    snapshots.append(personal_backtest(receipts))
personal = pd.concat(snapshots).reindex(core.index)
assert personal.index.is_unique and personal.quarter_comparisons.notna().all()

### Проверка переноса между месяцами

Февральские прогнозы обучаются только на январе, мартовские — на январе и феврале. Для каждого месяца сравниваем две полные аудитории из 5 000 клиентов. В столбцах `added_buyers` и `dropped_buyers` указаны покупатели, которых модель добавила и исключила.

In [4]:
report = []
for cutoff in sorted(train.scoring_date.unique())[1:]:
    fit_mask, held = train.scoring_date < cutoff, train.scoring_date == cutoff
    fit_ids, query_ids = train.customer_id.loc[fit_mask], pd.Index(train.customer_id.loc[held])
    name = str(cutoff.date())
    previous_groups = score_stage(name, fit_mask, query_ids, transition_train.loc[held],
                                  full_train.loc[held])['combined_regional']
    prior = pd.Series(train_stage(name, fit_ids, train_motifs.loc[query_ids], previous_groups), index=query_ids)
    scores = context_forecast(fit_ids, query_ids, prior)
    quarters = context_forecast(fit_ids, query_ids, prior, personal)
    y_held = labels.loc[query_ids]
    for model_name, prediction_scores in [('motifs', prior), ('context', scores), ('quarters', quarters)]:
        report.append(dict(date=name, model=model_name, **compare(y_held, prediction_scores, prior, core.n_orders)))
    rows = pd.DataFrame(report)
    save_csv(rows[rows.model != 'quarters'], 'validation_context.csv')
    save_csv(rows, 'validation_quarters.csv')
print(pd.DataFrame(report).to_string(index=False))

      date    model     ap20  buyers  changed  added_buyers  dropped_buyers  false_7_15
2022-02-01   motifs 0.774831    4159        0             0               0         333
2022-02-01  context 0.776896    4166      135            97              90         338
2022-02-01 quarters 0.777011    4166      136            94              87         338
2022-03-01   motifs 0.782566    4169        0             0               0         321
2022-03-01  context 0.784056    4170      108            65              64         322
2022-03-01 quarters 0.784343    4172      108            65              62         317


### Апрельский прогноз

Переобучаем на трёх доступных месяцах. Проверяем идентификаторы, порядок строк, диапазон вероятностей и содержимое ZIP.

In [5]:
def write_submission(scores, filename):
    submission = sample[['customer_id']].join(scores, on='customer_id', validate='one_to_one')
    assert scores.index.is_unique and set(scores.index) == set(sample.customer_id) == set(test.customer_id)
    assert submission.customer_id.equals(sample.customer_id) and submission.score.between(0,1).all()
    csv_path = Path(filename + '.csv')
    save_csv(submission, csv_path)
    pd.testing.assert_frame_equal(submission, pd.read_csv(csv_path))
    with ZipFile(filename + '.zip', 'w') as archive:
        entry = ZipInfo(csv_path.name)
        archive.writestr(entry, csv_path.read_bytes(), compress_type=ZIP_DEFLATED)
    with ZipFile(filename + '.zip') as archive:
        assert archive.namelist() == [csv_path.name]
        assert archive.read(csv_path.name) == csv_path.read_bytes()
    return submission

test_ids = pd.Index(test.customer_id)
reference = pd.read_csv('submission_motifs.csv').set_index('customer_id').score.reindex(test_ids)
result = context_forecast(train.customer_id, test_ids, reference)
submission_context = write_submission(result, 'submission_context')
quarter_scores = context_forecast(train.customer_id, test_ids, reference, personal)
submission_quarters = write_submission(quarter_scores, 'submission_quarters')
print(len(submission_quarters), 'клиентов в submission_quarters.csv')

25000 клиентов в submission_quarters.csv
